## PDF Extraction and Embedding (Programme Documents)

### 1. Extracting raw PDF data

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader, UnstructuredPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import os, re, pickle, copy

In [ ]:
eee_path = "https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/"
pdf_path = "../RAG/ProgramBooklet"

path = os.path.join(pdf_path)
pdfs = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]

#pdfs = pdfs[1:2] # For testing, only process the first PDF. Remove this line to process all PDFs.

llm = ChatOllama(model="llama3.1:8b", validate_model_on_init=True, temperature=0.3) # Changed model to LLaMA 3.1 8B.
print(pdfs)

['BEngBSc_Scheme_IAIE_46409_2526.pdf']


### Please install "poppler" and "tesseract" in Homebrew before proceeding to the next step!
Warning! Long running time! (Approx. 8-9 mins per documents)
<br>Last run-time: 59 mins

In [3]:
def regex_enhance(txt):
    text = re.sub(r' {2,}', ' ', txt)  # Strip excess white spaces
    return text

def extract_programme_title(first_page_content, llm_model):
    """Extract programme title from first page using LLM."""
    content_snippet = first_page_content[:2000]
    
    prompt = \
        f"""
        You are extracting the programme title from a university programme booklet's first page.

        *First page content*:
        {content_snippet}

        Extract the full programme title including:
        1. Degree type (e.g., Bachelor of Engineering, Master of Science, PhD)
        2. Discipline (e.g., Electrical Engineering, Electronic and Information Engineering)

        Return only the following:
        1. The programme title with normalized capitalizations
        2. The short form of such title (e.g., BEng / MSc in EE / EIE)
        Do NOT include any additional texts.

        The programme title and its short form in one line, separated by " | ":
        """
    
    # Extract programme title from first page using LLM
    try:
        response = llm_model.invoke(prompt)
        title = response.content.strip() 
        title = re.sub(r'\s+', ' ', title)
        title = title.replace('"', '').replace("'", "")

        if len(title) > 10 and len(title) < 250:
            return title
        else:
            return "Unknown"
    except Exception as e:
        print(f"LLM extraction failed: {e}")
        return "Unknown"

text_docs = []
raw_table_docs = []

unwanted_metadata = ["producer", "creator", "creationdate", "file_path", 
                     "format", "title", "subject", "keywords", "moddate", 
                     "author", "trapped", "modDate", "creationDate"]

for pdf in pdfs:
    # Extract programme title from first page
    loader = PyMuPDFLoader(f"{pdf_path}/{pdf}")
    cur_pdf = loader.load()
    programme_title = ""
    if len(cur_pdf) > 0:
        programme_title = extract_programme_title(cur_pdf[0].page_content, llm)
        print(f"PDF: {pdf} -> Programme: {programme_title}")
    
    # Load with Unstructured for elements
    loader = UnstructuredPDFLoader(
        f"{pdf_path}/{pdf}", 
        mode="elements", 
        strategy="hi_res", 
        infer_table_structure=True,
    )
    print(f"Loading {pdf}...")
    elements = loader.load()
    
    for element in elements:
        # Augment metadata
        element.metadata["source"] = pdf
        element.metadata["content_type"] = "pdf"
        element.metadata["programme_title"] = programme_title
        for key in unwanted_metadata:
            if key in element.metadata:
                del element.metadata[key]
        
        if element.metadata.get("category") == "Table":
            raw_table_docs.append(element)
        else:
            text_docs.append(element)

PDF: BEngBSc_Scheme_IAIE_46409_2526.pdf -> Programme: BEng/BSc (Hons) Scheme in Information and Artificial Intelligence Engineering | BEng/BSc (Hons) in IAIE
Loading BEngBSc_Scheme_IAIE_46409_2526.pdf...


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


Warning! Long running time! (Approx. 2 mins per table)
<br>Last run-time: 589 mins

In [4]:
# Table summarization
def summarize_table_element(element):
    print(f"Progress: Summarizing table from {element.metadata['source']}...")
    
    prompt = f"Summarize the following table in natural language in high detail:\n\n{element.page_content}"
    summary = llm.invoke(prompt).content.strip()
    element.metadata["table_content"] = element.page_content  # store original table
    element.page_content = summary
    element.metadata["content_type"] = "table_summary"
    return element

print(f"\nSummarizing {len(raw_table_docs)} tables...")

with ThreadPoolExecutor() as executor:
    table_docs = list(executor.map(summarize_table_element, raw_table_docs))

print("Table summarization complete.\n")


Summarizing 356 tables...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IAIE_46409_2526.pdf...
Progress: Summarizing table from BEngBSc_Scheme_IA

In [5]:
print(f"Number of text elements: {len(text_docs)}, table elements: {len(table_docs)}")
# print(text_docs[0].page_content if text_docs else "No text docs")

Number of text elements: 5105, table elements: 356


### Additional Step: For storing the table docs into pickle object (and loading it) for emergency cases

In [6]:
file_name = "table_docs.pkl"

# Export table_docs to a pickle file
with open(file_name, 'wb') as f:
    pickle.dump(table_docs, f)
print(f"Exported {len(table_docs)} table documents to {file_name}")

Exported 356 table documents to table_docs.pkl


In [7]:
# Import from the file
with open(file_name, 'rb') as f:
    loaded_table_docs = pickle.load(f)
print(f"Loaded {len(loaded_table_docs)} table documents from {file_name}")

# Verify
print(f"Original: {type(table_docs[0]) if table_docs else 'N/A'}")
print(f"Loaded: {type(loaded_table_docs[0]) if loaded_table_docs else 'N/A'}")
print(f"Match?: {table_docs == loaded_table_docs if table_docs and loaded_table_docs else 'N/A'}")

Loaded 356 table documents from table_docs.pkl
Original: <class 'langchain_core.documents.base.Document'>
Loaded: <class 'langchain_core.documents.base.Document'>
Match?: True


### 2. Text Splitting

In [8]:
print("table docs example:")
for i, doc in enumerate(table_docs[:1]): 
    print(f"Table doc {i}: {doc}")

table docs example:
Table doc 0: page_content='Here is a detailed summary of the table in natural language:

**Program Overview**

The university offers several undergraduate programs related to Information and Artificial Intelligence Engineering. The programs are designed to equip students with the necessary knowledge, skills, and competencies to succeed in their chosen careers.

**Program Details**

There are three main programs offered: BEng (Hons) / BSc (Hons) Scheme in Information and Artificial Intelligence Engineering, BEng (Hons) in Electronic Systems and Internet-of-Things, BSc (Hons) in Artificial Intelligence and Information Engineering, and BSc (Hons) in Information Security.

**Program Rationale**

Each program has a unique rationale that justifies its existence. For example, the BEng (Hons) / BSc (Hons) Scheme in Information and Artificial Intelligence Engineering aims to provide students with a comprehensive understanding of both theoretical and practical aspects of info

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(text_docs)

# Combine text chunks and table summaries using copy.deepcopy to avoid modifying original documents!!!
all_chunks = []
all_chunks = [copy.deepcopy(chunk) for chunk in chunks] + [copy.deepcopy(doc) for doc in table_docs]

for i, chunk in enumerate(all_chunks):
    source = chunk.metadata.get("source", "N/A")
    page = chunk.metadata.get("page_number", "N/A")
    programme = chunk.metadata.get("programme_title", "N/A")
    chunk_type = "table" if chunk.metadata.get("content_type") == "table_summary" else "chunk"
    chunk.metadata["chunk_id"] = f"PolyU_Doc_{source}_page_{page}_{chunk_type}_{i}"

    chunk.page_content = f"--- Source: {source}, Programme: {programme} --- \n --- Retrieved from: {eee_path} --- \n\n{chunk.page_content}"

    if "programme_title" in chunk.metadata:
        del chunk.metadata["programme_title"]
    if "coordinates" in chunk.metadata:
        del chunk.metadata["coordinates"]
    if "languages" in chunk.metadata:
        del chunk.metadata["languages"]

In [11]:
# Print a table chunk example
print(all_chunks[500])

page_content='--- Source: BEngBSc_Scheme_IAIE_46409_2526.pdf, Programme: BEng/BSc (Hons) Scheme in Information and Artificial Intelligence Engineering | BEng/BSc (Hons) in IAIE --- 
 --- Retrieved from: https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/ --- 

# Category: COM: Compulsory' metadata={'source': 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'last_modified': '2025-10-27T13:11:43', 'filetype': 'application/pdf', 'page_number': 29, 'file_directory': '../RAG/ProgramBooklet', 'filename': 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'parent_id': '9e9eafdbb6106f2b92a5aff18e12e4a3', 'category': 'UncategorizedText', 'element_id': 'de1f1d4d35c7409c76252cf47a744800', 'content_type': 'pdf', 'chunk_id': 'PolyU_Doc_BEngBSc_Scheme_IAIE_46409_2526.pdf_page_29_chunk_500'}


### 3. Document Embedding in Chroma

In [12]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [13]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

0

In [14]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, all_chunks))

In [15]:
for i, chunk in enumerate(all_chunks):
    print(f"Adding chunk {i+1}/{len(all_chunks)} to ChromaDB. Metadata: {chunk.metadata}\n")
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i+3000)]
    )

print(f"Added {len(all_chunks)} chunks into ChromaDB")

Adding chunk 1/5462 to ChromaDB. Metadata: {'source': 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'last_modified': '2025-10-27T13:11:43', 'filetype': 'application/pdf', 'page_number': 1, 'file_directory': '../RAG/ProgramBooklet', 'filename': 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'category': 'UncategorizedText', 'element_id': '3256e430606a8b10e729148ff902c8e2', 'content_type': 'pdf', 'chunk_id': 'PolyU_Doc_BEngBSc_Scheme_IAIE_46409_2526.pdf_page_1_chunk_0'}

Adding chunk 2/5462 to ChromaDB. Metadata: {'source': 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'last_modified': '2025-10-27T13:11:43', 'filetype': 'application/pdf', 'page_number': 1, 'file_directory': '../RAG/ProgramBooklet', 'filename': 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'category': 'UncategorizedText', 'element_id': '3f1490694e24d203ab28c5b7a518bd65', 'content_type': 'pdf', 'chunk_id': 'PolyU_Doc_BEngBSc_Scheme_IAIE_46409_2526.pdf_page_1_chunk_1'}

Adding chunk 3/5462 to ChromaDB. Metadata: {'source': 'BEngBSc_Scheme_IAIE_46409_2526.pd

### 4. Simple Testing

In [16]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What courses are offerred in the first year of IAIE programme?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content[:]}...")
    print(f"Source: {result.metadata.get('source')}, Page: {result.metadata.get('page')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: --- Source: BEngBSc_Scheme_IAIE_46409_2526.pdf, Programme: BEng/BSc (Hons) Scheme in Information and Artificial Intelligence Engineering | BEng/BSc (Hons) in IAIE --- 
 --- Retrieved from: https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/ --- 

Here is a detailed summary of the table in natural language:

**Year 1: Foundation Year**

The first year of study consists of two semesters, each with a combination of academic and training credits. The total number of academic credits for the year is 31, while the total number of training credits is 2.

In Semester 1, students take a range of subjects that provide a foundation in mathematics, science, and engineering. These include:

* Basic Mathematics I - Calculus (AMA1110) and Basic Mathematics II - Calculus and Probability & Statistics (AMA1120)
* Linear Algebra (3 credits)
* English Language and Communication (ELCXXXX) subject 1
* Tomorrow's Leaders (APSS1L01), a leadership development cours

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_23408/2722163706.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(
